In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, TensorDataset, DataLoader
from datasets import load_dataset
import matplotlib.pyplot as plt
from IPython.display import clear_output
import os

c:\Users\Mezon\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Load a subset of the TinyStories dataset (to keep it simple and manageable)
ds = load_dataset("roneneldan/TinyStories", split="train[:10000]")  # Use first 10000 stories for training
text = ' . '.join(ds['text'])  # Concatenate stories with ' . ' as separator

In [ ]:
# Building vocabulary
vocab = sorted(set(text + '.'))  # Include '.' as EOS
vocab_size = len(vocab)
stoi = {v: k for k, v in enumerate(vocab)}
itos = {v: k for k, v in stoi.items()}

def decode(seq):
    return ''.join([itos.get(i, '?') for i in seq])  # Handle unknown indices

def encode(name):
    return [stoi.get(s, 0) for s in name]  # Default to 0 if char not in vocab

In [ ]:
# Custom BatchNorm1d as in the notebook
class BatchNorm1d(nn.Module):
    def __init__(self, dim, momentum=0.1):
        super().__init__()
        self.dim = dim
        self.momentum = momentum
        self.eps = 1e-6

        self.scale = nn.Parameter(torch.ones(dim))
        self.shift = nn.Parameter(torch.zeros(dim))

        self.running_mean = torch.zeros(dim).to(device)
        self.running_var = torch.ones(dim).to(device)

    def forward(self, x):
        if self.training:
            xmean = x.mean(dim=0, keepdim=True)
            xvar = x.var(dim=0, keepdim=True)
        else:
            xmean = self.running_mean
            xvar = self.running_var

        x = (x - xmean) / (xvar + self.eps) ** 0.5

        x = self.scale * x + self.shift

        if self.training:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean.squeeze()
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar.squeeze()

        return x

In [ ]:
# Linear with Tanh and BatchNorm
class LinWithTanh(nn.Module):
    def __init__(self, in_features, out_features, bias=True, drop_rate=0.0):
        super().__init__()
        self.lin = nn.Linear(in_features, out_features, bias=bias)
        self.bn = BatchNorm1d(out_features)
        self.dropout = nn.Dropout(p=drop_rate)
        self.tanh = nn.Tanh()

    def forward(self, x):
        x = self.lin(x)
        x = self.bn(x)
        x = self.dropout(x)
        x = self.tanh(x)
        return x

In [ ]:
# Model similar to the notebook, adapted for stories
class StoryGenerator(nn.Module):
    def __init__(self, vocab_size, n_embd, block_size, n_hidden):
        super().__init__()
        self.E = nn.Embedding(vocab_size, n_embd)

        self.layer1 = LinWithTanh(block_size * n_embd, n_hidden)
        self.layers = nn.Sequential(
            *[LinWithTanh(n_hidden, n_hidden) for _ in range(5)]  # Fewer layers for simplicity
        )

        self.out = nn.Linear(n_hidden, vocab_size)
        self.apply(self._initialize)

    def _initialize(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.xavier_uniform_(module.weight)

    def forward(self, x):
        xemb = self.E(x)  # (B, T, C)
        B, T, C = xemb.shape
        x = xemb.view(B, T * C)  # (B, T*C)
        x = self.layer1(x)
        x = self.layers(x)
        logits = self.out(x)
        return logits

In [ ]:
# Hyperparameters
block_size = 8
n_embd = 32
n_hidden = 256  # Larger hidden for better capacity

In [ ]:
# Build dataset
encoded_text = encode(text)
X = []
Y = []

context = [0] * block_size
for ix in encoded_text:
    X.append(context)
    Y.append(ix)
    context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

# Split train/test
train_size = int(0.8 * X.shape[0])
Xtr, Xts = X[:train_size], X[train_size:]
Ytr, Yts = Y[:train_size], Y[train_size:]

Dtr = TensorDataset(Xtr, Ytr)
Dts = TensorDataset(Xts, Yts)

DLtr = DataLoader(Dtr, batch_size=1024, shuffle=True, drop_last=True)  # Larger batch for GPU
DLts = DataLoader(Dts, batch_size=1024, shuffle=False, drop_last=False)

In [ ]:
# Initialize model and optimizer
model = StoryGenerator(vocab_size, n_embd, block_size, n_hidden).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Training function
@torch.no_grad()
def evaluate(model):
    model.eval()
    epoch_loss = 0.0
    for x, y in DLts:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        epoch_loss += loss.item()
    epoch_loss /= len(DLts)
    return epoch_loss

def train(model, optimizer, n_epoch=5):  # Fewer epochs for simplicity
    lossi = []
    for epoch in range(n_epoch):
        model.train()
        epoch_loss = 0.0
        for x, y in DLtr:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            lossi.append(loss.item())
        epoch_loss /= len(DLtr)
        eval_loss = evaluate(model)
        print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.3f} | Eval Loss: {eval_loss:.3f}")
    return lossi

In [ ]:
# Train the model
lossi = train(model, optimizer)

Epoch 1 | Train Loss: 1.173 | Eval Loss: 1.065
Epoch 2 | Train Loss: 1.017 | Eval Loss: 1.013
Epoch 3 | Train Loss: 0.979 | Eval Loss: 0.987
Epoch 4 | Train Loss: 0.957 | Eval Loss: 0.977
Epoch 5 | Train Loss: 0.943 | Eval Loss: 0.964


In [ ]:
# Generation function
@torch.no_grad()
def generate_story(model, prompt="Once upon a time", max_len=500):
    model.eval()
    context = [0] * block_size
    prompt_enc = encode(prompt)
    # Ensure context is exactly block_size
    if len(prompt_enc) > block_size:
        prompt_enc = prompt_enc[-block_size:]
    context = [0] * (block_size - len(prompt_enc)) + prompt_enc
    out = prompt_enc.copy()

    for _ in range(max_len):
        input_tensor = torch.tensor([context]).to(device)
        logits = model(input_tensor)
        probs = F.softmax(logits[-1], dim=-1)  # Sample from the last position
        ix = torch.multinomial(probs, num_samples=1).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == stoi.get('.', 0):  # Stop at EOS if encountered
            break

    return decode(out)

In [ ]:
print("Generated Story:")
print(generate_story(model, prompt="One day"))

Generated Story:
One day Jacky got from then on, everyone who wanted to pretty!"
Joe stopped to earlier.


In [ ]:
# Saving model and related data
torch.save({
    'model_state_dict': model.state_dict(),
    'vocab': vocab,
    'stoi': stoi,
    'itos': itos,
    'block_size': block_size,
    'n_embd': n_embd,
    'n_hidden': n_hidden,
}, 'tinystories_uz.pth')

print("Model saqlandi: tinystories_uz.pth")

Model saqlandi: tinystories_uz.pth
